# 4. SBI Parameter Recovery & Inference (BE & SC)

Fit both BE and SC models to behavioural data using Simulation-Based Inference.

**Workflow**:
1. Single-session parameter recovery for both models
2. Compare recovery quality: which model's parameters are better identifiable?
3. Posterior predictive checks
4. Multi-session setup (GP-linked)


In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import warnings, time
warnings.filterwarnings('ignore')

from Models.BE_core import BEParams, BEState, BEModel
from Models.SC_core import SCParams, SCState, SCModel
from Analysis.summary_stats import compute_summary_stats, get_stat_names_expanded
from Data.structures import (
    generate_synthetic_animal, sample_stimuli, stimulus_to_category,
    param_trajectory_naive_to_expert, sc_param_trajectory_naive_to_expert,
)
from Inference.simulator import (
    Simulator, SimulatorConfig, ModelType,
    create_be_simulator, create_sc_simulator,
    get_sbi_prior, wrap_for_sbi,
)

try:
    import torch
    from sbi.inference import SNPE
    from sbi.utils import BoxUniform
    SBI_AVAILABLE = True
    print("SBI + PyTorch available")
except ImportError:
    SBI_AVAILABLE = False
    print("SBI not available. Install with: pip install sbi torch")

plt.rcParams['figure.dpi'] = 100

## 1. Single-Session Parameter Recovery

In [ ]:
# --- Configuration ---
STAT_NAMES = [
    'accuracy', 'psychometric', 'recency', 'win_stay',
    'stimulus_sensitivity', 'logistic_history',
]
expanded = get_stat_names_expanded(STAT_NAMES)
n_stats = len(expanded)
print(f"Using {n_stats} stats: {expanded}")

N_TRIALS = 300
BURN_IN = 100
N_SBI_SIMS = 10_000
N_TEST = 50
SEED = 42

# Generate shared stimuli
rng_stim = np.random.default_rng(SEED)
stimuli = sample_stimuli(N_TRIALS, 'Uniform', rng_stim)
categories = stimulus_to_category(stimuli)

In [ ]:
# --- Build simulators using the Simulator class ---
be_sim = create_be_simulator(
    stimuli, categories,
    stat_names=STAT_NAMES,
    burn_in=BURN_IN,
    seed=SEED,
)

sc_sim = create_sc_simulator(
    stimuli, categories,
    stat_names=STAT_NAMES,
    burn_in=BURN_IN,
    seed=SEED,
)

print(f"BE: {be_sim.n_free_params} free params: {be_sim.get_param_names()}")
print(f"SC: {sc_sim.n_free_params} free params: {sc_sim.get_param_names()}")

# Quick test
theta_be = be_sim.sample_prior(seed=42)
stats_be = be_sim(theta_be)
print(f"\nBE test: theta={theta_be}, stats shape={stats_be.shape}")

theta_sc = sc_sim.sample_prior(seed=42)
stats_sc = sc_sim(theta_sc)
print(f"SC test: theta={theta_sc}, stats shape={stats_sc.shape}")

In [ ]:
# --- SBI training for both models ---
def train_sbi(simulator, model_name, n_sims=N_SBI_SIMS):
    if not SBI_AVAILABLE:
        print(f"Skipping {model_name} — SBI not available")
        return None, None

    print(f"\n{'='*60}")
    print(f"Training SBI for {model_name} ({n_sims} simulations)...")
    t0 = time.time()

    prior = get_sbi_prior(simulator)
    sbi_sim = wrap_for_sbi(simulator)

    inference = SNPE(prior=prior)
    theta_train = prior.sample((n_sims,))
    x_train = torch.stack([sbi_sim(t) for t in theta_train])

    # Remove NaN simulations
    valid = ~torch.any(torch.isnan(x_train), dim=1)
    theta_train = theta_train[valid]
    x_train = x_train[valid]
    print(f"  {valid.sum()}/{n_sims} valid simulations")

    _ = inference.append_simulations(theta_train, x_train)
    density_estimator = inference.train()
    posterior = inference.build_posterior(density_estimator)

    print(f"  Training took {time.time()-t0:.1f}s")
    return posterior, prior

be_posterior, be_prior = train_sbi(be_sim, 'BE')
sc_posterior, sc_prior = train_sbi(sc_sim, 'SC')

### Recovery Test

In [ ]:
def recovery_test(simulator, posterior, prior, model_name, n_test=N_TEST):
    if posterior is None:
        return None, None, None

    sbi_sim = wrap_for_sbi(simulator)
    rng_test = np.random.default_rng(123)
    param_names = simulator.get_param_names()

    true_list, mean_list, std_list = [], [], []

    for i in range(n_test):
        true_theta = prior.sample((1,)).squeeze()
        x_obs = sbi_sim(true_theta)

        if torch.any(torch.isnan(x_obs)):
            continue

        samples = posterior.sample((500,), x=x_obs)
        true_list.append(true_theta.numpy())
        mean_list.append(samples.mean(0).numpy())
        std_list.append(samples.std(0).numpy())

    return np.array(true_list), np.array(mean_list), np.array(std_list)


be_true, be_mean, be_std = recovery_test(be_sim, be_posterior, be_prior, 'BE')
sc_true, sc_mean, sc_std = recovery_test(sc_sim, sc_posterior, sc_prior, 'SC')

In [ ]:
# --- Recovery plots side by side ---
if be_true is not None or sc_true is not None:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle('Parameter Recovery: BE (top) vs SC (bottom)',
                 fontsize=13, fontweight='bold')

    for row, (true_arr, mean_arr, std_arr, sim, colour, model_name) in enumerate([
        (be_true, be_mean, be_std, be_sim, 'steelblue', 'BE'),
        (sc_true, sc_mean, sc_std, sc_sim, 'darkorange', 'SC'),
    ]):
        if true_arr is None:
            for ax in axes[row]:
                ax.text(0.5, 0.5, 'N/A', transform=ax.transAxes, ha='center')
            continue

        param_names = sim.get_param_names()
        for p, (ax, label) in enumerate(zip(axes[row], param_names)):
            tv = true_arr[:, p]
            pv = mean_arr[:, p]
            pe = std_arr[:, p]

            valid = ~np.isnan(pv)
            ax.errorbar(tv[valid], pv[valid], yerr=pe[valid],
                         fmt='o', ms=3, alpha=0.5, color=colour, elinewidth=0.5)

            lims = [min(tv[valid].min(), pv[valid].min()),
                    max(tv[valid].max(), pv[valid].max())]
            ax.plot(lims, lims, 'k--', lw=0.8, alpha=0.5)

            r, pval = spearmanr(tv[valid], pv[valid])
            ax.set_title(f'{label}\nr={r:.2f}, p={pval:.3f}', fontsize=9)
            ax.set_xlabel('True', fontsize=8)
            if p == 0:
                ax.set_ylabel(f'{model_name}\nRecovered', fontsize=9)
            ax.tick_params(labelsize=7)

    plt.tight_layout()
    plt.show()

## 2. Posterior Predictive Checks

In [ ]:
if SBI_AVAILABLE and (be_posterior is not None):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, posterior, sim, model_name, colour in [
        (axes[0], be_posterior, be_sim, 'BE', 'steelblue'),
        (axes[1], sc_posterior, sc_sim, 'SC', 'darkorange'),
    ]:
        if posterior is None:
            ax.text(0.5, 0.5, 'N/A', transform=ax.transAxes, ha='center')
            continue

        sbi_sim = wrap_for_sbi(sim)
        prior = get_sbi_prior(sim)
        true_theta = prior.sample((1,)).squeeze()
        x_obs = sbi_sim(true_theta).numpy()

        post_samples = posterior.sample((200,), x=torch.tensor(x_obs, dtype=torch.float32))
        x_pred = []
        for s in post_samples:
            x = sbi_sim(s)
            if not torch.any(torch.isnan(x)):
                x_pred.append(x.numpy())

        if x_pred:
            x_pred = np.array(x_pred)
            stat_names = get_stat_names_expanded(STAT_NAMES)

            n_show = min(8, len(stat_names))
            x_pos = np.arange(n_show)
            ax.boxplot(x_pred[:, :n_show], positions=x_pos, widths=0.5,
                        boxprops=dict(color=colour),
                        medianprops=dict(color='black'))
            ax.scatter(x_pos, x_obs[:n_show], marker='*', s=100,
                        color='red', zorder=5, label='Observed')
            ax.set_xticks(x_pos)
            ax.set_xticklabels(stat_names[:n_show], rotation=45, fontsize=7, ha='right')
            ax.set_title(f'{model_name}: Posterior Predictive', fontsize=10)
            ax.legend()

    plt.tight_layout()
    plt.show()

## 3. Multi-Session Setup

For fitting session trajectories (GP-linked learning rate), use the
`Simulator` class with `varying_params`.

In [ ]:
# --- Multi-session demo (setup only, not trained) ---
# Generate multi-session stimuli
n_sessions = 5
multi_stim = np.column_stack([
    sample_stimuli(N_TRIALS, 'Uniform', np.random.default_rng(SEED + s))
    for s in range(n_sessions)
])
multi_cats = np.where(multi_stim >= 0, 1, 0)

# BE: vary eta_learning across sessions
be_multi_config = SimulatorConfig(
    model_type=ModelType.BE,
    varying_params=['eta_learning'],
    stat_names=STAT_NAMES,
    burn_in=BURN_IN,
    state_transition='identity',
)
be_multi_sim = Simulator(be_multi_config, multi_stim, multi_cats, seed=SEED)
print(f"BE multi-session: {be_multi_sim.n_free_params} free params")
print(f"  Params: {be_multi_sim.get_param_names()}")

# SC: vary gamma across sessions
sc_multi_config = SimulatorConfig(
    model_type=ModelType.SC,
    varying_params=['gamma'],
    stat_names=STAT_NAMES,
    burn_in=BURN_IN,
    state_transition='identity',
)
sc_multi_sim = Simulator(sc_multi_config, multi_stim, multi_cats, seed=SEED)
print(f"\nSC multi-session: {sc_multi_sim.n_free_params} free params")
print(f"  Params: {sc_multi_sim.get_param_names()}")

# Test forward simulation
theta_be_multi = be_multi_sim.sample_prior(seed=42)
stats_be_multi = be_multi_sim(theta_be_multi)
print(f"\nBE multi-session test: theta shape={theta_be_multi.shape}, stats shape={stats_be_multi.shape}")

theta_sc_multi = sc_multi_sim.sample_prior(seed=42)
stats_sc_multi = sc_multi_sim(theta_sc_multi)
print(f"SC multi-session test: theta shape={theta_sc_multi.shape}, stats shape={stats_sc_multi.shape}")

## Summary

**Recovery comparison**: Which model's parameters are better identifiable?

- If BE has tighter posteriors and higher correlations → BE parameters are more
  distinguishable, suggesting the boundary representation provides a natural
  parameterisation for this task.
- If SC matches or exceeds BE → the category-distribution representation may be
  equally valid, offering an alternative interpretation of the same behaviour.

**For real data**: Replace `sample_stimuli` with actual experimental stimuli and
the rest of the pipeline works identically. Fit both models to the same data
and compare via posterior predictive checks and model comparison metrics.
